### Exploration du dataset et idées pour le modèle de prédiction

In [24]:
#Charger les libraries nécessaires
import sys
import os
import duckdb
# Add the project root to the Python path
sys.path.append(os.path.abspath('..'))

import pandas as pd
from utilities import customdataloader

Bloc réutilisable pour le loading année par année

In [7]:
# Ajuster ce chemin selon votre setup Google Drive Desktop
DATA_ROOT = r"G:\.shortcut-targets-by-id\1SR7TxhNQjFyS8f2bbuo9pcLvzNMLBMwY\CSD\data"

In [21]:
# Instancier le loader
loader = customdataloader(
    pricefile=os.path.join(DATA_ROOT, 'prices', 'prices.parquet'),
    costfile=os.path.join(DATA_ROOT, 'costs', 'costs.parquet'),
    dsimfile=[os.path.join(DATA_ROOT, 'sim_daily', f'sim_daily_{y}.parquet') for y in [2020,2021,2022,2023]],
    msimfile=[os.path.join(DATA_ROOT, 'sim_monthly', f'sim_monthly_{y}.parquet') for y in [2020,2021,2022,2023]],
)

In [9]:
# Chargement complet de costs + prices (ces fichiers sont petits)
costs = pd.read_parquet(os.path.join(DATA_ROOT, 'costs', 'costs.parquet'))
prices = pd.read_parquet(os.path.join(DATA_ROOT, 'prices', 'prices.parquet'))

print(f"costs:  {costs.shape[0]:,} rows  |  prices: {prices.shape[0]:,} rows")

costs:  9,092 rows  |  prices: 453,167 rows


In [30]:
# Explorer un mois spécifique
# Choisir le mois cible et le cutoff correspondant
# Choisir le mois cible et le cutoff correspondant
TARGET_MONTH = '2020-02'       # M+1 : le mois qu'on veut prédire
CUTOFF_DATE  = '2020-01-07'    # 7 de M : moment de décision
 
# Trouver le bon fichier sim_monthly selon l'année cible
target_year = TARGET_MONTH[:4]
sim_file = os.path.join(DATA_ROOT, 'sim_monthly', f'sim_monthly_{target_year}.parquet')
 
# Construire filterdf à partir des EIDs présents dans les simulations du mois cible
sim_eids = duckdb.query(f"""
    SELECT DISTINCT EID
    FROM read_parquet('{sim_file}')
    WHERE strftime(DATETIME, '%Y-%m') = '{TARGET_MONTH}'
""").to_df()
 
known_eids = sim_eids['EID'].values
filterdf = pd.DataFrame({
    'EID': list(known_eids) * 2,
    'MONTH': [TARGET_MONTH] * len(known_eids) * 2,
    'PEAKID': [0] * len(known_eids) + [1] * len(known_eids),
})
print(f"Candidats pour {TARGET_MONTH}: {len(filterdf)} triplets ({len(known_eids)} EIDs uniques)")

Candidats pour 2020-02: 9076 triplets (4538 EIDs uniques)


In [31]:
filterdf.head(5) # Afficher les 5 premiers triplets candidats

,EID,MONTH,PEAKID
0,1386,2020-02,0
1,1409,2020-02,0
2,1426,2020-02,0
3,1428,2020-02,0
4,1430,2020-02,0


In [32]:
#Simulations mensuelles (pour le mois cible)
monthly_sim = loader.get_monthly_data(filterdf)
print(f"Sim monthly: {monthly_sim.shape}")
monthly_sim.head()

Sim monthly: (0, 13)


,SCENARIOID,EID,DATETIME,PEAKID,m_ACTIVATIONLEVEL,m_WINDIMPACT,m_SOLARIMPACT,m_HYDROIMPACT,m_NONRENEWBALIMPACT,m_EXTERNALIMPACT,m_LOADIMPACT,m_TRANSMISSIONOUTAGEIMPACT,m_PSM
